# Berlin building classification -- Graph Transformer (Path D)

A real Graph Transformer over Berlin's building stock, alongside (not replacing) the
`DecisionTreeClassifier` baseline in `berlin_all.ipynb` (`clf2_berlin`, 91.9% accuracy,
0.79 non-residential F1). Reuses the same feature table
(`data/processed/berlin_buildings_features.parquet`) and the same feature columns
(`src.prepare_data.feature_columns`) so results are directly comparable to that baseline --
only the model changes.

**Scale note**: trained on a 60,000-building stratified random subsample of the full 331k
ALKIS-labeled set, spread across all of Berlin (not restricted to one borough). Full-batch
attempts on all 331k nodes OOM-killed this process repeatedly (grew to 9-12GB and got killed
by the kernel, taking other running apps down with it each time) -- full-batch GNN training
materializes a per-edge attention tensor on every forward pass, which doesn't fit this
laptop's memory at that scale. The correct scalable fix is mini-batch/neighbor-sampling
training (`torch_geometric.loader.NeighborLoader`); subsampling plus a small model is the
fast, reliable substitute for tonight, not the intended long-term approach.

**Graph construction**: nodes = buildings, edges = k-nearest neighbors on building centroids
(EPSG:25833, so distances are in meters). Transductive node classification -- one graph,
train/val/test are boolean masks over nodes, not separate row subsets.

**Model**: `torch_geometric.nn.TransformerConv` (the UniMP-style graph transformer attention
layer -- Shi et al. 2021), which restricts attention to real graph edges rather than doing
full O(n^2) self-attention. Even at 60k nodes, full self-attention (~3.6x10^9 pairs) would be
impractical; this is why a graph-structured attention mechanism is the right choice here
rather than a plain sequence Transformer, independent of the memory-driven subsampling above.

**Results (last successful run)**: 91.06% test accuracy, 0.76 non-residential F1, 0.85 macro
F1 -- close to but slightly below the `clf2_berlin` tree baseline (91.9%/0.79/0.87), which is
reasonable given this trained on 1/5.5th the data with a deliberately small architecture.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.utils import to_undirected

from src.prepare_data import feature_columns

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [2]:
buildings = gpd.read_parquet("../data/processed/berlin_buildings_features.parquet")
labeled_full = buildings.dropna(subset=["alkis_label"]).reset_index(drop=True)

# Subsample -- full-batch attempts at all 331k nodes OOM-killed this process repeatedly;
# full-batch GNN training materializes a per-edge attention tensor every forward pass, which
# doesn't fit this laptop's memory at that scale. train_test_split(..., stratify=...) gives a
# clean stratified subsample -- simpler and more robust than a groupby/apply/sample approach,
# which turned out to drop the grouping column in this pandas version.
SAMPLE_SIZE = 60_000
sample_idx, _ = train_test_split(
    np.arange(len(labeled_full)), train_size=SAMPLE_SIZE, random_state=42,
    stratify=labeled_full["alkis_label"],
)
labeled = labeled_full.iloc[sample_idx].reset_index(drop=True)

feature_cols = feature_columns(labeled)
print(len(feature_cols), "features,", len(labeled), "labeled buildings (subsampled from", len(labeled_full), ")")

X_raw = labeled[feature_cols].to_numpy(dtype=np.float32)
y_raw = (labeled["alkis_label"] == "non_residential").to_numpy(dtype=np.int64)  # 0=residential, 1=non_residential
centroids = labeled.geometry.centroid
coords = np.column_stack([centroids.x.to_numpy(), centroids.y.to_numpy()])  # EPSG:25833, meters

print("class balance:", np.bincount(y_raw))

86 features, 60000 labeled buildings (subsampled from 331168 )
class balance: [48215 11785]


In [3]:
# Graph construction: k-nearest-neighbor edges on building centroids (meters, EPSG:25833).
# Cheap and memory-safe compared to the polygon buffer/overlay operations in
# src/prepare_data.py -- this is just a KD-tree query over point coordinates, not repeated
# geometry buffering. K=5 (not larger) to keep edge count -- and therefore the attention
# layer's memory -- low.
K = 5
nn = NearestNeighbors(n_neighbors=K + 1).fit(coords)  # +1 because a point is its own nearest neighbor
_, indices = nn.kneighbors(coords)

src_nodes = np.repeat(np.arange(len(coords)), K)
dst_nodes = indices[:, 1:].reshape(-1)  # drop self-loops (column 0)
edge_index = torch.tensor(np.vstack([src_nodes, dst_nodes]), dtype=torch.long)
edge_index = to_undirected(edge_index, num_nodes=len(coords))
print("edges:", edge_index.shape[1])

edges: 367564


In [4]:
# Stratified train/val/test node masks -- 70/10/20, matching the 80/20 test split used for
# clf2_berlin as closely as practical while keeping a validation slice to monitor training.
n = len(y_raw)
idx = np.arange(n)
train_idx, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_raw)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1, random_state=42, stratify=y_raw[train_idx])

# Fit scaler on train rows only (avoid leakage), apply to all nodes.
scaler = StandardScaler().fit(X_raw[train_idx])
X_scaled = scaler.transform(X_raw).astype(np.float32)

x = torch.tensor(X_scaled, dtype=torch.float)
y = torch.tensor(y_raw, dtype=torch.long)
train_mask = torch.zeros(n, dtype=torch.bool); train_mask[train_idx] = True
val_mask = torch.zeros(n, dtype=torch.bool); val_mask[val_idx] = True
test_mask = torch.zeros(n, dtype=torch.bool); test_mask[test_idx] = True

data = Data(x=x, edge_index=edge_index, y=y,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask).to(device)
print(data)

Data(x=[60000, 86], edge_index=[2, 367564], y=[60000], train_mask=[60000], val_mask=[60000], test_mask=[60000])


In [5]:
class BuildingGraphTransformer(torch.nn.Module):
    """UniMP-style Graph Transformer (Shi et al. 2021): TransformerConv restricts
    self-attention to real graph edges instead of the full O(n^2) pairwise attention a
    plain sequence Transformer would need.

    hidden_channels/heads kept deliberately small (16/2, 2 layers) -- TransformerConv
    materializes a per-edge message tensor of shape [E, heads, hidden_channels], and a wider
    64/4/3-layer config OOM-killed this process repeatedly even before subsampling. This is
    the standard full-batch-GNN memory trap; the correct scalable fix is mini-batch/neighbor
    sampling (torch_geometric.loader.NeighborLoader), not attempted here given time
    constraints -- a small model is the fast, reliable fix for tonight.
    """
    def __init__(self, in_channels, hidden_channels=16, heads=2, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.conv2 = TransformerConv(hidden_channels * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


model = BuildingGraphTransformer(in_channels=x.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# Same 2:1 non_residential:residential class weighting used for clf2_berlin, for consistency.
class_weights = torch.tensor([1.0, 2.0], dtype=torch.float, device=device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
print(model)

BuildingGraphTransformer(
  (conv1): TransformerConv(86, 16, heads=2)
  (conv2): TransformerConv(32, 2, heads=1)
)


In [6]:
def evaluate(mask):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        pred = logits[mask].argmax(dim=1)
        true = data.y[mask]
        acc = (pred == true).float().mean().item()
        f1_macro = f1_score(true.cpu(), pred.cpu(), average="macro")
    return acc, f1_macro


EPOCHS = 60
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0 or epoch == 1:
        val_acc, val_f1 = evaluate(data.val_mask)
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | val_acc {val_acc:.4f} | val_macro_f1 {val_f1:.4f}")

epoch   1 | loss 0.7512 | val_acc 0.7404 | val_macro_f1 0.6463


epoch   5 | loss 0.4864 | val_acc 0.8646 | val_macro_f1 0.7733


epoch  10 | loss 0.4053 | val_acc 0.8925 | val_macro_f1 0.8093


epoch  15 | loss 0.3641 | val_acc 0.8996 | val_macro_f1 0.8279


epoch  20 | loss 0.3432 | val_acc 0.9073 | val_macro_f1 0.8461


epoch  25 | loss 0.3297 | val_acc 0.9106 | val_macro_f1 0.8510


... (3 outputs left out)

epoch  45 | loss 0.3079 | val_acc 0.9148 | val_macro_f1 0.8582


epoch  50 | loss 0.3050 | val_acc 0.9154 | val_macro_f1 0.8598


epoch  55 | loss 0.3006 | val_acc 0.9158 | val_macro_f1 0.8601


epoch  60 | loss 0.3006 | val_acc 0.9158 | val_macro_f1 0.8601


In [7]:
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    test_pred = logits[data.test_mask].argmax(dim=1).cpu().numpy()
    test_true = data.y[data.test_mask].cpu().numpy()

print("Accuracy:", accuracy_score(test_true, test_pred))
print(classification_report(test_true, test_pred, target_names=["residential", "non_residential"]))
print(confusion_matrix(test_true, test_pred))

print("\nFor reference -- clf2_berlin (DecisionTreeClassifier) baseline from berlin_all.ipynb:")
print("Accuracy: 0.9189, non_residential F1: 0.79, macro F1: 0.87")

Accuracy: 0.9105833333333333
                 precision    recall  f1-score   support

    residential       0.93      0.96      0.95      9643
non_residential       0.81      0.71      0.76      2357

       accuracy                           0.91     12000
      macro avg       0.87      0.83      0.85     12000
   weighted avg       0.91      0.91      0.91     12000

[[9258  385]
 [ 688 1669]]

For reference -- clf2_berlin (DecisionTreeClassifier) baseline from berlin_all.ipynb:
Accuracy: 0.9189, non_residential F1: 0.79, macro F1: 0.87
